# Code Demo Đồ án: Giảm chiều và Nhận dạng Khuôn mặt với Fisherfaces (PCA + LDA)
**Bộ dữ liệu:** Olivetti Faces (AT&T Laboratories Cambridge)
**Môn học:** Toán cho Machine Learning / Cơ sở Trí tuệ Nhân tạo

Notebook này cài đặt thủ công và kiểm chứng quy trình toán học của thuật toán **Fisherfaces** (kết hợp PCA và LDA để giải quyết bài toán suy biến ma trận *High-Dimensional, Small Sample Size - HDSSS*) theo đúng lý thuyết tài liệu chuẩn.

In [ ]:
# Import các thư viện cần thiết
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_olivetti_faces
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns

# Kiểm tra phiên bản và thiết lập hiển thị
print("Các thư viện đã được nạp thành công!")

--- 
## 1. Nạp và Khám phá Dữ liệu Olivetti Faces
- **Số lớp ($C$)**: 40 người khác nhau.
- **Số ảnh mỗi người ($N_c$)**: 10 ảnh.
- **Tổng số mẫu ($N$)**: 400 ảnh.
- **Kích thước ảnh gốc**: $64 \times 64$ pixel (đã resize, ảnh xám), số chiều đặc trưng $d = 64 \times 64 = 4096$.

In [ ]:
# Nạp bộ dữ liệu Olivetti Faces từ scikit-learn
olivetti = fetch_olivetti_faces(shuffle=True, random_state=42)
X = olivetti.images  # Shape: (400, 64, 64)
y = olivetti.target  # Shape: (400,)

n_samples, h, w = X.shape
d = h * w
C = len(np.unique(y))

print(f"Tổng số mẫu (N): {n_samples}")
print(f"Số chiều đặc trưng ban đầu (d): {d} (64x64)")
print(f"Số lớp / số người (C): {C}")
print(f"Số mẫu mỗi lớp (Nc): {n_samples // C}")

# Vector hóa dữ liệu: mỗi hàng là một ảnh vector d-chiều -> Chuyển vị thành ma trận (d x N) theo lý thuyết
X_flat = X.reshape((n_samples, d))
X_matrix = X_flat.T  # Kích thước (4096, 400)

--- 
## 2. Chia tập Huấn luyện (Train) và Kiểm tra (Test)
Sử dụng `stratified split` để đảm bảo mỗi lớp đều xuất hiện cân đối ở cả tập train và test.

In [ ]:
import random

# Chia tỉ lệ 70% train, 30% test có phân tầng theo nhãn
random_state = random.randint(0, 4294967295)
print(f"Random state được sử dụng: {random_state}")
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_flat, y, test_size=0.3, stratify=y, random_state=random_state
)

print(f"Số mẫu huấn luyện: {X_train_raw.shape[0]}")
print(f"Số mẫu kiểm tra: {X_test_raw.shape[0]}")

--- 
## 3. Tiền xử lý: Căn giữa dữ liệu (Mean-Centering)
Tính vector trung bình $\mu$ trên tập huấn luyện và trừ đi trên cả tập huấn luyện và kiểm tra để tránh rò rỉ dữ liệu (*data leakage*).

In [ ]:
# Tính global mean trên tập train (shape: (d, 1))
mu_train = np.mean(X_train_raw, axis=0).reshape(-1, 1)

# Căn giữa dữ liệu tập train và test
X_train_centered = X_train_raw.T - mu_train  # (4096, N_train)
X_test_centered = X_test_raw.T - mu_train    # (4096, N_test)

print("Shape của tập huấn luyện sau khi căn giữa:", X_train_centered.shape)

--- 
## 4. Bước 1 của Fisherfaces: PCA Sơ bộ (Eigenfaces)
Do $d = 4096 \gg N_{\text{train}} \approx 280$, ma trận phân tán trong lớp $S_W$ bị suy biến. Ta dùng PCA để giảm số chiều trung gian xuống $m = N_{\text{train}} - C$ (hoặc nhỏ hơn) nhằm khử không gian vô nghĩa (null space).

In [ ]:
N_train = X_train_centered.shape[1]
# Số chiều PCA trung gian tối ưu theo lý thuyết: m = N_train - C
m_pca = N_train - C 
print(f"Số chiều PCA trung gian được chọn (m = N_train - C): {m_pca}")

# Áp dụng thủ thuật Turk-Pentland thông qua SVD của X_train_centered (U, S, Vt)
# X_train_centered = U * S * Vt
U, S, Vt = np.linalg.svd(X_train_centered, full_matrices=False)

# Lấy m_pca vector riêng đầu tiên làm ma trận chiếu W_pca (d x m_pca)
W_pca = U[:, :m_pca]
print(
    "Kích thước ma trận chiếu W_pca:", W_pca.shape
)  # (4096, m_pca)

--- 
## 5. Bước 2 của Fisherfaces: LDA trong không gian con PCA
Chiếu dữ liệu huấn luyện xuống không gian $m$ chiều, sau đó tính toán ma trận phân tán trong lớp $S_W'$ và giữa các lớp $S_B'$ để giải bài toán trị riêng tổng quát.

In [ ]:
# Chiếu tập train xuống không gian PCA: kích thước (m_pca, N_train)
X_train_pca = W_pca.T @ X_train_centered

# Tính toán các thành phần phân tán trong không gian con PCA
classes = np.unique(y_train)
S_W_prime = np.zeros((m_pca, m_pca))
overall_mean_pca = np.mean(X_train_pca, axis=1, keepdims=True)

S_B_prime = np.zeros((m_pca, m_pca))

for c in classes:
    # Lấy các mẫu thuộc lớp c trong không gian PCA
    X_c = X_train_pca[:, y_train == c]
    N_c = X_c.shape[1]
    
    # Vector trung bình của lớp c trong không gian PCA
    mu_c_pca = np.mean(X_c, axis=1, keepdims=True)
    
    # Cộng dồn ma trận phân tán trong lớp S_W'
    diff_W = X_c - mu_c_pca
    S_W_prime += diff_W @ diff_W.T
    
    # Cộng dồn ma trận phân tán giữa các lớp S_B'
    diff_B = mu_c_pca - overall_mean_pca
    S_B_prime += N_c * (diff_B @ diff_B.T)

# Giải bài toán trị riêng tổng quát: (S_W'^(-1) * S_B') w = lambda * w
# Sử dụng hàm linalg.inv vì S_W' đã khả nghịch trong không gian m_pca = N_train - C
S_W_inv = np.linalg.inv(S_W_prime)
eigenvalues, eigenvectors = np.linalg.eig(S_W_inv @ S_B_prime)

# Sắp xếp các trị riêng theo thứ tự giảm dần
sorted_indices = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[sorted_indices]
eigenvectors = eigenvectors[:, sorted_indices]

# Số chiều tối đa của LDA bị chặn trên bởi C - 1 = 39
max_lda_dim = C - 1
W_lda = eigenvectors[:, :max_lda_dim].real

print("Kích thước ma trận chiếu W_lda:", W_lda.shape)  # (m_pca, 39)

--- 
## 6. Bước 3: Kết hợp và Chiếu dữ liệu cuối cùng (Fisherfaces)
Tính ma trận chiếu tổng hợp $W_{\text{fisher}} = W_{\text{pca}} W_{\text{lda}}$ và thực hiện chiếu toàn bộ dữ liệu.

In [ ]:
# Ma trận chiếu Fisherfaces cuối cùng (d x 39)
W_fisher = W_pca @ W_lda
print("Kích thước ma trận chiếu Fisherfaces tổng hợp (W_fisher):", W_fisher.shape)

# Chiếu tập train và test lên không gian Fisherfaces
Y_train_fisher = W_fisher.T @ X_train_centered
Y_test_fisher = W_fisher.T @ X_test_centered

print(
    "Shape dữ liệu sau khi chiếu (Train):", Y_train_fisher.shape
)  # (39, N_train)
print("Shape dữ liệu sau khi chiếu (Test):", Y_test_fisher.shape)      # (39, N_test)

--- 
## 7. Phân loại (Nearest Neighbor) và Đánh giá Mô hình
Sử dụng quy tắc khoảng cách Euclid tối thiểu (Nearest Neighbor) trên không gian Fisherfaces để phân loại ảnh kiểm tra.

In [ ]:
# Thực hiện phân loại Nearest Neighbor thủ công trên không gian Fisherfaces
y_pred = []
for i in range(Y_test_fisher.shape[1]):
    test_sample = Y_test_fisher[:, i:i+1]
    # Tính khoảng cách Euclid từ mẫu test đến toàn bộ các mẫu train
    distances = np.linalg.norm(Y_train_fisher - test_sample, axis=0)
    # Tìm nhãn của mẫu huấn luyện có khoảng cách ngắn nhất
    nearest_idx = np.argmin(distances)
    y_pred.append(y_train[nearest_idx])

y_pred = np.array(y_pred)

# Đánh giá độ chính xác
acc = accuracy_score(y_test, y_pred)
print(f"Random state được sử dụng: {random_state}")
print(f"Độ chính xác (Accuracy) của mô hình Fisherfaces tự cài đặt: {acc * 100:.2f}%")

## 7.1. Kiểm tra độ ổn định với 10 random_state ngẫu nhiên
Chạy lại pipeline 10 lần với các cách chia train/test khác nhau để quan sát dải accuracy thực tế của mô hình.

In [ ]:
import random
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier

accuracies = []
print("=" * 55)
print("KIỂM TRA ĐỘ ỔN ĐỊNH VỚI 10 RANDOM_STATE NGẪU NHIÊN")
print("=" * 55)

for i in range(10):
    rs = random.randint(0, 4294967295)
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_flat, y, test_size=0.3, stratify=y, random_state=rs
    )
    
    # Mean-centering đúng như mục 3
    mu = np.mean(X_tr, axis=0)
    X_tr_c = X_tr - mu
    X_te_c = X_te - mu

    # Tính động n_components theo N_train của lần chia này
    n_components = X_tr_c.shape[0] - C  # m = N_train - C

    pipe = Pipeline([
        ('pca', PCA(n_components=n_components)),
        ('lda', LinearDiscriminantAnalysis()),
        ('knn', KNeighborsClassifier(n_neighbors=1))
    ])
    pipe.fit(X_tr_c, y_tr)
    acc = pipe.score(X_te_c, y_te)
    accuracies.append(acc)
    print(f"  Lần {i+1:2d} | random_state={rs:<12} | n_pca={n_components} | Accuracy: {acc*100:.2f}%")

print("-" * 55)
print(f"  Trung bình    : {np.mean(accuracies)*100:.2f}%")
print(f"  Tốt nhất      : {np.max(accuracies)*100:.2f}%")
print(f"  Tệ nhất       : {np.min(accuracies)*100:.2f}%")
print(f"  Độ lệch chuẩn : {np.std(accuracies)*100:.2f}%")
print("=" * 55)

--- 
## 8. Trực quan hóa Không gian Fisherfaces (2D Scatter Plot)
Chiếu dữ liệu xuống 2 thành phần Fisherfaces đầu tiên để quan sát khả năng phân tách giữa các lớp khuôn mặt.

In [ ]:
# Lấy 2 chiều đầu tiên của không gian Fisherfaces cho tập train
plt.figure(figsize=(10, 8))
scatter = plt.scatter(
    Y_train_fisher[0, :],
    Y_train_fisher[1, :],
    c=y_train,
    cmap='tab20',
    alpha=0.8,
    edgecolor='k',
    s=40
)
plt.title('Trực quan hóa Không gian Fisherfaces (2 chiều đầu tiên)')
plt.xlabel('Fisherface 1')
plt.ylabel('Fisherface 2')
plt.colorbar(scatter, label='Subject ID (Classes)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

--- 
## 9. So sánh nhanh với scikit-learn (LinearDiscriminantAnalysis)
Kiểm chứng kết quả bằng thư viện chuẩn `sklearn.discriminant_analysis.LinearDiscriminantAnalysis` (sử dụng `solver='svd'`).

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA

# Sử dụng pipeline LDA của sklearn trực tiếp trên tập train đã giảm chiều bằng PCA
lda_sklearn = LDA(solver='svd')
X_train_lda_sk = lda_sklearn.fit_transform(X_train_pca.T, y_train)
# X_test_lda_sk = lda_sklearn.transform(W_pca.T @ X_test_centered)
X_test_pca_transposed = (W_pca.T @ X_test_centered).T
X_test_lda_sk = lda_sklearn.transform(X_test_pca_transposed)

# Phân loại đơn giản bằng Nearest Neighbor hoặc dùng score tích hợp
# lda_sklearn.fit(X_train_pca.T, y_train)
# acc_sklearn = lda_sklearn.score(W_pca.T @ X_test_centered, y_test)
acc_sklearn = lda_sklearn.score(X_test_pca_transposed, y_test)
print(f"Độ chính xác sử dụng sklearn LDA (trên không gian PCA): {acc_sklearn * 100:.2f}%")

## 10. Đánh giá mô hình thực sự bằng Cross-Validation (10-fold)

Để đánh giá khách quan hơn, ta dùng **Stratified 10-fold Cross-Validation** thay vì một lần chia train/test cố định. Mỗi fold sẽ dùng 1/10 dữ liệu làm tập test, đảm bảo mỗi lớp đều xuất hiện cân đối. Kết quả **trung bình** qua 10 fold mới là độ chính xác đại diện thực sự của mô hình.

In [ ]:
# ============================================================
# 10. Đánh giá mô hình thực sự bằng Cross-Validation (10-fold)
# ============================================================
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier

# Pipeline tương đương với pipeline thủ công ở các bước 4-7
pipe = Pipeline([
    ('pca', PCA(n_components=240)),
    ('lda', LinearDiscriminantAnalysis()),
    ('knn', KNeighborsClassifier(n_neighbors=1))
])

# 10-fold Stratified: đảm bảo mỗi fold đều có đủ 40 lớp
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=None)
scores = cross_val_score(pipe, X_flat, y, cv=cv)

print("=" * 50)
print("KẾT QUẢ CROSS-VALIDATION (10-fold)")
print("=" * 50)
for i, s in enumerate(scores, 1):
    print(f"  Fold {i:2d}: {s*100:.2f}%")
print("-" * 50)
print(f"  Trung bình : {scores.mean()*100:.2f}%")
print(f"  Độ lệch chuẩn: {scores.std()*100:.2f}%")
print(f"  Tốt nhất   : {scores.max()*100:.2f}%")
print(f"  Tệ nhất    : {scores.min()*100:.2f}%")
print("=" * 50)